# BKV VP1 Subtyping Pipeline

Complete de novo pipeline for sub-typing BK polyomavirus VP1 fragments.


## Step 0 — Input QC & Translation

In [ ]:
from pathlib import Path
from Bio.Seq import Seq
from Bio.Align import PairwiseAligner
import re, json, urllib.request, urllib.parse, time
from collections import Counter, defaultdict
from io import StringIO
from Bio import SeqIO
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Phylo import draw_ascii, NewickIO
QUERY = """--paste your sequence here--""".replace("\n", "").upper()
gc = (QUERY.count('G') + QUERY.count('C')) / len(QUERY) * 100
best_frame, best_aa, best_stops = None, None, 999
for f in (1, 2, 3):
    sub = QUERY[f-1:]
    sub = sub[:(len(sub) // 3) * 3]
    aa = str(Seq(sub).translate())
    stops = aa.count('*')
    if stops < best_stops:
        best_frame, best_aa, best_stops = f, aa, stops
query_aa = best_aa
query_nt_aligned = QUERY[1:] if best_frame == 2 else QUERY
Path("outputs").mkdir(parents=True, exist_ok=True)
Path("outputs/query.fasta").write_text(f">query\n{QUERY}\n")
Path("outputs/query_translated.fasta").write_text(f">query_protein\n{query_aa}\n")
len(QUERY), gc, best_frame, len(query_aa)


## Step 1 — BLASTn vs NCBI nr/nt

In [ ]:
def run_blast(seq, db="nt", max_hits=50, timeout=540):
    params = {
        "CMD": "Put", "PROGRAM": "blastn", "DATABASE": db,
        "QUERY": f">query\n{seq}", "MEGABLAST": "on",
        "HITLIST_SIZE": str(max_hits), "EXPECT": "10",
    }
    url = "https://blast.ncbi.nlm.nih.gov/Blast.cgi"
    data = urllib.parse.urlencode(params).encode()
    req = urllib.request.Request(url, data=data, method="POST")
    resp = urllib.request.urlopen(req, timeout=60)
    html = resp.read().decode("utf-8", "replace")
    rid = re.search(r"RID = (\S+)", html).group(1)
    for attempt in range(36):
        time.sleep(15)
        check = urllib.parse.urlencode({
            "CMD": "Get", "RID": rid,
            "FORMAT_OBJECT": "SearchInfo", "FORMAT_TYPE": "Text",
        }).encode()
        try:
            resp = urllib.request.urlopen(
                urllib.request.Request(url, data=check, method="POST"), timeout=60)
        except:
            continue
        txt = resp.read().decode("utf-8", "replace")
        m = re.search(r"Status=(\S+)", txt)
        if m and m.group(1) == "READY":
            get = urllib.parse.urlencode({
                "CMD": "Get", "RID": rid, "FORMAT_TYPE": "Text",
                "ALIGNMENTS": str(max_hits), "DESCRIPTIONS": str(max_hits),
                "SHOW_OVERVIEW": "false", "FORMAT_OBJECT": "Alignment",
            }).encode()
            resp = urllib.request.urlopen(
                urllib.request.Request(url, data=get, method="POST"), timeout=120)
            return resp.read().decode("utf-8", "replace")
    return None
blast_report = run_blast(QUERY)
if blast_report:
    Path("outputs/blast_report.txt").write_text(blast_report)
"done" if blast_report else "BLAST timed out"


## Step 2 — Load Reference Panel

In [ ]:
def extract_vp1(gb_path):
    rec = next(SeqIO.parse(StringIO(gb_path.read_text()), "genbank"))
    src = next((f for f in rec.features if f.type == "source"), None)
    geno = None
    if src:
        for note in src.qualifiers.get("note", []):
            m = re.search(r"genotype:\s*(\S+)", note)
            if m:
                geno = m.group(1)
                break
        if not geno:
            geno = src.qualifiers.get("genotype", [None])[0]
    for f in rec.features:
        if f.type != "CDS":
            continue
        p = f.qualifiers.get("product", [""])[0]
        g = f.qualifiers.get("gene", [""])[0]
        n = f.qualifiers.get("note", [""])[0]
        if "VP1" in p or "VP1" in g or "VP1" in n:
            vp1_nt = str(f.extract(rec.seq)) if f.location else None
            return (geno, f.qualifiers.get("translation", [None])[0], vp1_nt)
    return (geno, None, None)
panel = {}
gb_dirs = [
    Path("references"),
    Path(),
]
for d in gb_dirs:
    if not d.exists():
        continue
    for gb in d.glob("*.gb"):
        info = extract_vp1(gb)
        geno, prot, nt = info
        if geno and prot and nt and len(prot) == 362 and len(nt) >= 1086:
            panel[gb.stem] = (geno, prot, nt)
counter = Counter(v[0] for v in panel.values())
dict(counter)


## Step 3 — Multiple Sequence Alignment (column-wise)

In [ ]:
SLICE_AA_S, SLICE_AA_E = 41, 124
SLICE_NT_S, SLICE_NT_E = 123, 372
genotype_I_subs = ["Ia", "Ib1", "Ib2", "Ic"]
all_subs = ["Ia", "Ib1", "Ib2", "Ic", "II", "III", "IV", "I"]
variable_positions = []
per_pos = {}
for i in range(249):
    per_sub = {s: set() for s in all_subs}
    for acc, (sub, prot, nt) in panel.items():
        if sub in all_subs and len(nt) >= SLICE_NT_E:
            per_sub[sub].add(nt[SLICE_NT_S:SLICE_NT_E][i].upper())
    per_pos[i] = per_sub
    all_res = set()
    for r in per_sub.values():
        all_res |= r
    if len(all_res) > 1:
        variable_positions.append(i)
len(variable_positions)


## Step 4 — NJ Tree

In [ ]:
records = []
for acc, (sub, prot, nt) in panel.items():
    records.append(SeqIO.SeqRecord(
        Seq(prot[SLICE_AA_S:SLICE_AA_E]),
        id=f"{sub}_{acc}", description=""))
records.append(SeqIO.SeqRecord(Seq(query_aa), id=">>>QUERY", description=""))
alignment = MultipleSeqAlignment(records)
calc = DistanceCalculator("identity")
dm = calc.get_distance(alignment)
constructor = DistanceTreeConstructor()
tree = constructor.nj(dm)
buf = StringIO()
NewickIO.write([tree], buf)
Path("outputs/tree.newick").write_text(buf.getvalue())
sorted_dists = sorted([
    (dm[">>>QUERY", n], n) for n in dm.names if n != ">>>QUERY"
])
[(round(d, 4), n) for d, n in sorted_dists[:10]]


## Step 5 Pairwise Distance Matrix (per-sub-lineage nt match)

In [ ]:
matches = {s: 0 for s in genotype_I_subs}
mismatches = {s: [] for s in genotype_I_subs}
for i in variable_positions:
    q = query_nt_aligned[i]
    for sub in genotype_I_subs:
        residues = set()
        for acc, (sub_, prot, nt) in panel.items():
            if sub_ == sub and len(nt) >= SLICE_NT_E:
                residues.add(nt[SLICE_NT_S:SLICE_NT_E][i].upper())
        if q in residues:
            matches[sub] += 1
        else:
            mismatches[sub].append(i + 1)
total = len(variable_positions)
results = {}
for sub in genotype_I_subs:
    results[sub] = {
        "match": f"{matches[sub]}/{total}",
        "pct": round(100 * matches[sub] / total, 1),
        "mismatches": mismatches[sub],
    }
results


## Step 6 — Subtype-Diagnostic Residue Scan

In [ ]:
exact_per_sub = defaultdict(int)
total_per_sub = defaultdict(int)
for acc, (sub, prot, nt) in panel.items():
    total_per_sub[sub] += 1
    if prot[SLICE_AA_S:SLICE_AA_E] == query_aa:
        exact_per_sub[sub] += 1
protein_results = {}
for sub in all_subs:
    if total_per_sub[sub]:
        protein_results[sub] = f"{exact_per_sub[sub]}/{total_per_sub[sub]}"
protein_results


In [ ]:
key_positions = [1, 12, 82, 95, 123, 165, 222, 237, 246]
diag = []
for i0 in key_positions:
    i = i0 - 1
    vp1_res = 42 + (i // 3)
    cs = (i // 3) * 3
    q_codon = query_nt_aligned[cs:cs+3]
    q_aa_res = str(Seq(q_codon).translate())
    row = {
        "pos": i0,
        "VP1_res": vp1_res,
        "query_nt": query_nt_aligned[i],
        "query_codon": q_codon,
        "query_aa": q_aa_res,
    }
    for sub in genotype_I_subs:
        row[sub] = "/".join(sorted(per_pos[i][sub]))
    diag.append(row)
diag


In [ ]:
novel = []
for i in range(249):
    q = query_nt_aligned[i]
    if q not in "ATCG":
        continue
    all_ref = set()
    for acc, (sub, prot, nt) in panel.items():
        if sub in genotype_I_subs and len(nt) >= SLICE_NT_E:
            all_ref.add(nt[SLICE_NT_S:SLICE_NT_E][i].upper())
    if q not in all_ref:
        codon_num = (i // 3) + 1
        vp1_res = 41 + codon_num
        cs = (i // 3) * 3
        q_cod = query_nt_aligned[cs:cs+3]
        q_aa_res = str(Seq(q_cod).translate())
        novel.append({
            "pos": i + 1,
            "VP1_res": vp1_res,
            "query_nt": q,
            "query_codon": q_cod,
            "query_aa": q_aa_res,
            "ref_residues": sorted(all_ref),
        })
novel


## Step 7 Recombination Test (skipped if fragment < 500 bp)

In [ ]:
if len(query_nt_aligned) < 500:
    recomb_result = "SKIPPED — fragment too short for RDP5/GARD"
else:
    recomb_result = "Run RDP5/GARD on the full VP1"
recomb_result


## Step 8 Final Call

In [ ]:
best_nt = max(genotype_I_subs, key=lambda s: matches[s])
best_protein = max(all_subs, key=lambda s: exact_per_sub.get(s, 0)) if any(exact_per_sub.values()) else "NONE"
has_novel = len(novel) > 0
concordant = all(
    matches[s] < matches[best_nt] or s == best_nt
    for s in genotype_I_subs if total_per_sub.get(s, 0) >= 10
) and not has_novel
if has_novel:
    confidence = "LOW"
    call = f"UNCLASSIFIABLE — {len(novel)} NOVEL substitution(s)"
elif concordant and matches[best_nt] == total:
    confidence = "HIGH"
    call = best_nt
else:
    confidence = "MODERATE"
    call = f"mosaic — nt-best={best_nt}, protein-best={best_protein}"
{"call": call, "confidence": confidence, "nt_best": best_nt, "protein_best": best_protein, "novel_count": len(novel)}


## Export Reference CSV

In [ ]:
import csv
VALID_NTS = set("ATCG")
geno_I = ["Ia", "Ib1", "Ib2", "Ic"]
geno_other = ["II", "III", "IV"]
comparison_groups = {}
for s in geno_I:
    comparison_groups[s] = [x for x in geno_I if x != s]
for s in geno_other:
    comparison_groups[s] = [x for x in geno_other if x != s]
comparison_groups["I"] = geno_I + geno_other
fps_per_sub = defaultdict(list)
for i in variable_positions:
    for sub in all_subs:
        comp = comparison_groups.get(sub, [])
        for res in per_pos[i][sub]:
            if res not in VALID_NTS:
                continue
            is_unique = any(res not in per_pos[i].get(o, set()) for o in comp)
            if is_unique:
                fps_per_sub[sub].append((i + 1, res))
refs_out = []
for acc, (sub, prot, nt) in sorted(panel.items()):
    trimmed = nt[SLICE_NT_S:SLICE_NT_E].upper()
    if len(trimmed) != 249:
        continue
    fps = fps_per_sub.get(sub, [])
    count = sum(1 for pos, res in fps if trimmed[pos-1] == res)
    refs_out.append([trimmed, sub, f"{count}/{len(fps)}" if fps else "0/0", acc])
csv_path = Path("reference_sequences_trimmed.csv")
with open(csv_path, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["trimmed_reference_sequence", "subtype",
                     "unique_subtype_fingerprints_count_out_of_total",
                     "accession_number"])
    writer.writerows(refs_out)
len(refs_out), str(csv_path)
